# Agentic AI for Maritime Freight Pricing and Route Optimization
### Infosys Springboard 7.0 — Milestone 1: User Authentication Portal
**Streamlit + JWT + SQLite (on Google Drive) + ngrok + Gmail OTP**

Run the cells **top to bottom**. Before the first run, add these in Colab → 🔑 **Secrets** and switch *Notebook access* ON for each:

| Secret | What it is |
|---|---|
| `JWT_SECRET` | any long random string (signs session tokens) |
| `NGROK_AUTHTOKEN` | token from your ngrok dashboard |
| `EMAIL_ADDRESS` | Gmail address that sends the OTP |
| `EMAIL_PASSWORD` | 16-character Gmail App Password |
| `ADMIN_USERNAME` | username for the admin login page |
| `ADMIN_PASSWORD` | password for the admin login page |


### 1️⃣ Install packages

In [1]:
!pip install -q streamlit==1.66.1 streamlit-option-menu pyngrok pyjwt bcrypt plotly pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 11.9 MB/s eta 0:00:00


### 2️⃣ Setup — Google Drive database + Colab Secrets

In [2]:
import os
from pathlib import Path
from google.colab import drive, userdata

# ── 1. Persistent database location (Google Drive) ───────────────────────────
# Users, hashed passwords and lock-out counters live in a SQLite file on your Drive,
# so they are still there after you re-run the notebook or the Colab runtime restarts.
try:
    drive.mount("/content/drive")
    DB_DIR = Path("/content/drive/MyDrive/Infosys_Portal")
except Exception as e:
    print("⚠️ Google Drive not mounted -> using temporary local storage "
          "(registered users will be lost when the runtime resets).", type(e).__name__)
    DB_DIR = Path("/content/infosys_portal_data")
DB_DIR.mkdir(parents=True, exist_ok=True)
os.environ["DB_PATH"] = str(DB_DIR / "infosys_portal.db")

# ── 2. Load every secret from Colab Secrets into the environment (values are never printed)
REQUIRED = ["JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_ADDRESS", "EMAIL_PASSWORD", "ADMIN_USERNAME", "ADMIN_PASSWORD"]
missing = []
for key in REQUIRED:
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if value:
        os.environ[key] = value
    else:
        missing.append(key)
    print(f"{key}: {'Configured ✅' if value else 'MISSING ❌'}")
if missing:
    raise RuntimeError(f"Add these Colab Secrets (and enable notebook access): {', '.join(missing)}")

# ── 3. Streamlit theme (must exist BEFORE the server starts)
Path(".streamlit").mkdir(exist_ok=True)
Path(".streamlit/config.toml").write_text(
    '[theme]\nbase="light"\nprimaryColor="#4f46e5"\nbackgroundColor="#f8fafc"\n'
    'secondaryBackgroundColor="#eef2ff"\ntextColor="#0f172a"\n\n[browser]\ngatherUsageStats=false\n'
)
print("\n✅ Setup complete. Database file:", os.environ["DB_PATH"])


⚠️ Google Drive not mounted -> using temporary local storage (registered users will be lost when the runtime resets). MessageError
JWT_SECRET: Configured ✅
NGROK_AUTHTOKEN: Configured ✅
EMAIL_ADDRESS: Configured ✅
EMAIL_PASSWORD: Configured ✅
ADMIN_USERNAME: Configured ✅
ADMIN_PASSWORD: Configured ✅

✅ Setup complete. Database file: /content/infosys_portal_data/infosys_portal.db


### 3️⃣ Write the Streamlit app (`app.py`)

In [3]:
%%writefile app.py
import os, re, hmac, hashlib, secrets, smtplib, sqlite3, time, datetime, html as _html
from contextlib import contextmanager
from email.message import EmailMessage

import bcrypt, jwt
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
from streamlit_option_menu import option_menu

st.set_page_config(page_title="Maritime AI Portal", page_icon="⚓", layout="wide", initial_sidebar_state="auto")

# ============================================================
# CONFIG  (every secret comes from environment variables that the
# notebook copies out of Colab Secrets -- nothing is hard-coded)
# ============================================================
JWT_SECRET = os.environ.get("JWT_SECRET")
ADMIN_USERNAME = os.environ.get("ADMIN_USERNAME")
ADMIN_PASSWORD = os.environ.get("ADMIN_PASSWORD")
DB_PATH = os.environ.get("DB_PATH", "infosys_portal.db")   # points to Google Drive when mounted

APP_NAME = "Maritime AI Portal"
PROJECT_TITLE = "Agentic AI for Maritime Freight Pricing and Route Optimization"
MAX_ATTEMPTS = 3        # wrong password / wrong OTP / wrong security answer attempts allowed
LOCK_SECONDS = 30       # how long the account stays locked
OTP_TTL = 300           # OTP validity in seconds
SECURITY_QUESTIONS = [
    "What is your pet name?",
    "What is your mother's maiden name?",
    "What is your favourite city?",
    "What was the name of your first school?",
    "What is your favourite food?",
]

if not JWT_SECRET:
    st.error("JWT_SECRET is not configured. Run the setup cell (it loads your Colab Secrets) before launching the app.")
    st.stop()

# ============================================================
# STYLING
# ============================================================
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
:root{--ink:#0f172a;--muted:#64748b;--line:#e2e8f0;--brand:#0369a1;--brand2:#0891b2;--soft:#e0f2fe;}
html, body, .stApp, [data-testid="stAppViewContainer"]{font-family:'Inter',sans-serif !important;color:var(--ink);}
.stApp{background:radial-gradient(900px 520px at 6% -8%,#e0f2fe 0%,transparent 60%),radial-gradient(800px 500px at 100% 0%,#ccfbf1 0%,transparent 55%),#f8fafc !important;}
[data-testid="stHeader"]{background:transparent !important;}
footer,#MainMenu,[data-testid="stDecoration"],[data-testid="stAppDeployButton"],.stDeployButton{display:none !important;}
[data-testid="stMainBlockContainer"],.block-container{max-width:1180px !important;padding:2.2rem 2rem 2rem !important;}
h1,h2,h3,h4{font-family:'Inter',sans-serif !important;color:var(--ink) !important;letter-spacing:-.02em;}
[data-testid="stWidgetLabel"] p,label p{font-weight:600 !important;font-size:13px !important;color:#334155 !important;}

/* ---------- hero panel ---------- */
.hero{position:relative;overflow:hidden;border-radius:28px;padding:40px 40px 0;color:#fff;
  background:linear-gradient(160deg,rgba(14,116,144,.55) 0%,rgba(8,47,73,.88) 70%);border:1px solid rgba(255,255,255,.14);
  box-shadow:0 40px 80px -30px rgba(0,0,0,.65);}
.hero > *{position:relative;z-index:1;}
.brand{display:flex;align-items:center;gap:12px;font-weight:700;font-size:17px;color:#e0f2fe;}
.brand .logo{width:40px;height:40px;border-radius:12px;background:rgba(34,211,238,.16);border:1px solid rgba(34,211,238,.35);display:flex;align-items:center;justify-content:center;font-size:20px;}
.chip{display:inline-block;margin-top:30px;padding:5px 12px;border-radius:999px;font-size:11.5px;font-weight:700;letter-spacing:.12em;text-transform:uppercase;color:#67e8f9;background:rgba(34,211,238,.12);border:1px solid rgba(34,211,238,.35);}
.hero h1{color:#fff !important;font-size:2.05rem !important;line-height:1.18;margin:14px 0 14px !important;font-weight:800;}
.hero h1 em{font-style:normal;background:linear-gradient(90deg,#67e8f9,#fcd34d);-webkit-background-clip:text;background-clip:text;color:transparent;-webkit-text-fill-color:transparent;}
.hero p.lead{color:#bae6fd;font-size:14.5px;line-height:1.65;margin:0;}
.feat{display:flex;gap:14px;align-items:flex-start;margin-top:18px;}
.feat .ic{flex:0 0 38px;height:38px;border-radius:10px;background:rgba(255,255,255,.10);border:1px solid rgba(255,255,255,.14);display:flex;align-items:center;justify-content:center;font-size:18px;}
.feat b{display:block;font-size:14px;color:#fff;}
.feat span{font-size:12.5px;color:#a5d8ef;}
.scene{display:block;width:calc(100% + 80px);margin:26px -40px 0;}

/* ---------- form card ---------- */
[data-testid="stForm"]{background:#fff;border:1px solid var(--line) !important;border-radius:24px !important;padding:34px 32px !important;box-shadow:0 25px 50px -22px rgba(15,23,42,.20);}
.card-title{font-size:26px;font-weight:800;margin:0;letter-spacing:-.02em;color:var(--ink);}
.card-sub{color:var(--muted);font-size:14px;margin:4px 0 20px;}

/* ---------- inputs ---------- */
div[data-baseweb="base-input"]{background:transparent !important;border:none !important;}
div[data-baseweb="input"],div[data-baseweb="select"] > div{background:#f8fafc !important;border:1.5px solid var(--line) !important;border-radius:12px !important;}
div[data-baseweb="input"]:focus-within,div[data-baseweb="select"] > div:focus-within{border-color:#0891b2 !important;box-shadow:0 0 0 4px rgba(8,145,178,.18) !important;background:#fff !important;}
input{color:var(--ink) !important;-webkit-text-fill-color:var(--ink) !important;font-size:14.5px !important;}
input::placeholder{color:#94a3b8 !important;-webkit-text-fill-color:#94a3b8 !important;}

/* ---------- buttons ---------- */
div[data-testid="stButton"] button,button[data-testid^="stBaseButton"]{border-radius:12px !important;height:46px;font-weight:700 !important;font-size:14px !important;transition:all .18s ease;}
button[data-testid="stBaseButton-primary"],button[data-testid="stBaseButton-primaryFormSubmit"],button[kind="primary"],button[kind="primaryFormSubmit"]{
  background:linear-gradient(135deg,#0c4a6e,#0891b2) !important;color:#fff !important;border:none !important;box-shadow:0 10px 20px -8px rgba(8,145,178,.65) !important;}
button[data-testid="stBaseButton-primary"]:hover,button[data-testid="stBaseButton-primaryFormSubmit"]:hover,button[kind="primary"]:hover,button[kind="primaryFormSubmit"]:hover{
  transform:translateY(-2px);box-shadow:0 14px 24px -8px rgba(8,145,178,.75) !important;}
button[data-testid="stBaseButton-secondary"],button[data-testid="stBaseButton-secondaryFormSubmit"],button[kind="secondary"],button[kind="secondaryFormSubmit"]{
  background:#fff !important;color:#0c4a6e !important;border:1.5px solid var(--line) !important;}
button[data-testid="stBaseButton-secondary"]:hover,button[data-testid="stBaseButton-secondaryFormSubmit"]:hover,button[kind="secondary"]:hover{
  border-color:#22d3ee !important;background:#ecfeff !important;}
[data-testid="stAlert"]{border-radius:12px !important;}

/* ---------- lock banner ---------- */
.lock{display:flex;align-items:center;gap:16px;border-radius:16px;padding:16px 18px;background:#fff1f2;border:1px solid #fecdd3;color:#9f1239;margin-top:12px;}
.lock .t{font-size:30px;font-weight:800;min-width:74px;text-align:center;background:#fff;border-radius:12px;padding:6px 8px;}
.lock b{display:block;font-size:15px;}
.lock span{font-size:13px;opacity:.9;}

/* ---------- dashboard ---------- */
section[data-testid="stSidebar"]{background:#fff !important;border-right:1px solid var(--line) !important;}
.banner{border-radius:20px;padding:26px 32px;display:flex;justify-content:space-between;align-items:center;gap:16px;margin-bottom:22px;
  background:linear-gradient(135deg,#04111d,#0c4a6e 60%,#0e7490);color:#fff;box-shadow:0 20px 40px -20px rgba(12,74,110,.7);}
.banner h1{color:#fff !important;margin:0 !important;font-size:1.55rem !important;font-weight:800;}
.banner .sub{color:#bae6fd;font-size:13.5px;margin-top:4px;}
.pill{background:rgba(255,255,255,.16);padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;white-space:nowrap;}
.kpi{background:#fff;border:1px solid var(--line);border-radius:18px;padding:20px;text-align:center;box-shadow:0 14px 28px -20px rgba(15,23,42,.25);}
.kpi .ic{font-size:26px;}
.kpi .v{font-size:26px;font-weight:800;color:var(--ink);margin-top:4px;}
.kpi .l{color:var(--muted);font-size:12px;font-weight:600;}
"""
st.markdown(f"<style>{CSS}</style>", unsafe_allow_html=True)

from urllib.parse import quote
_WAVE = ("<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 1440 220' preserveAspectRatio='none'>"
         "<path d='M0 90 C180 40 360 140 540 90 S900 40 1080 90 S1300 130 1440 80 V220 H0Z' fill='#06b6d4' fill-opacity='.10'/>"
         "<path d='M0 140 C200 100 380 190 600 140 S960 100 1160 150 S1360 170 1440 130 V220 H0Z' fill='#0891b2' fill-opacity='.16'/>"
         "<path d='M0 185 C240 160 420 215 700 185 S1100 160 1440 190 V220 H0Z' fill='#164e63' fill-opacity='.45'/></svg>")
_WAVE_URI = "data:image/svg+xml;utf8," + quote(_WAVE, safe="/:=;,'")
AUTH_BG_CSS = (".stApp{background:url(\"" + _WAVE_URI + "\") bottom/100% 240px no-repeat,"
               "radial-gradient(rgba(255,255,255,.055) 1px,transparent 1px) 0 0/30px 30px,"
               "radial-gradient(700px 420px at 88% 8%,rgba(6,182,212,.28),transparent 62%),"
               "radial-gradient(800px 520px at 0% 100%,rgba(14,116,144,.38),transparent 60%),"
               "linear-gradient(160deg,#030d17 0%,#07213a 55%,#0a3350 100%) !important;background-attachment:fixed !important;}"
               "[data-testid='stStatusWidget'],[data-testid='stStatusWidget'] *{color:#cbd5e1 !important;}")


def H(s):
    """Collapse indentation/newlines so markdown never turns HTML into a code block."""
    return " ".join(line.strip() for line in s.strip().splitlines())


def esc(s):
    return _html.escape(str(s), quote=True)


# ============================================================
# DATABASE  (SQLite file -- stored on Google Drive so it survives re-runs / restarts)
# ============================================================
@contextmanager
def db():
    conn = sqlite3.connect(DB_PATH, timeout=15)
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


@st.cache_resource
def init_db():
    os.makedirs(os.path.dirname(os.path.abspath(DB_PATH)), exist_ok=True)
    with db() as c:
        c.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
            password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
        cols = [r[1] for r in c.execute("PRAGMA table_info(users)")]
        if "pwd_fp" not in cols:                      # upgrade databases created by the old version
            c.execute("ALTER TABLE users ADD COLUMN pwd_fp TEXT")
        c.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_users_pwd_fp ON users(pwd_fp)")
        c.execute("""CREATE TABLE IF NOT EXISTS lockouts (
            key TEXT PRIMARY KEY, attempts INTEGER NOT NULL DEFAULT 0,
            locked_until REAL NOT NULL DEFAULT 0, last_try REAL NOT NULL DEFAULT 0)""")
        # the old version stored a hard-coded admin inside the users table -- remove it
        c.execute("DELETE FROM users WHERE email='infosys@ai'")
    return True


init_db()


@st.cache_resource
def dummy_hash():
    return bcrypt.hashpw(b"not-a-real-password", bcrypt.gensalt()).decode()


def hash_txt(t):
    return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()


def check_txt(t, h):
    try:
        return bool(h) and bcrypt.checkpw(t.encode(), h.encode())
    except ValueError:        # e.g. password longer than bcrypt's 72-byte limit
        return False


def pwd_fp(p):
    """Keyed fingerprint of a password. bcrypt hashes are salted so they can't be compared
    between users; this lets us detect 'another account already uses this password'."""
    return hmac.new(("pw-fp|" + JWT_SECRET).encode(), p.encode(), hashlib.sha256).hexdigest()


def password_taken(p, exclude_email=None):
    with db() as c:
        if exclude_email:
            r = c.execute("SELECT 1 FROM users WHERE pwd_fp=? AND email<>?", (pwd_fp(p), exclude_email)).fetchone()
        else:
            r = c.execute("SELECT 1 FROM users WHERE pwd_fp=?", (pwd_fp(p),)).fetchone()
    return bool(r)


def find_user(ident):
    with db() as c:
        c.row_factory = sqlite3.Row
        r = c.execute("SELECT * FROM users WHERE lower(email)=lower(?) OR lower(username)=lower(?)",
                      (ident, ident)).fetchone()
        return dict(r) if r else None


# ============================================================
# VALIDATION
# ============================================================
# >=2 letters before @, >=2 letters between @ and the last dot, >=2 letters after the last dot
EMAIL_RE = re.compile(r"[A-Za-z]{2,}[A-Za-z0-9._%+-]*@[A-Za-z]{2,}[A-Za-z0-9-]*(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}")
USERNAME_RE = re.compile(r"[A-Za-z0-9][A-Za-z0-9 ._-]{2,29}")


def valid_email(e):
    return bool(EMAIL_RE.fullmatch(e.strip()))


def valid_username(u):
    return bool(USERNAME_RE.fullmatch(u.strip()))


def password_problems(p):
    out = []
    if len(p) < 8: out.append("at least 8 characters")
    if len(p) > 64: out.append("at most 64 characters")
    if not any(c.isupper() for c in p): out.append("an uppercase letter")
    if not any(c.islower() for c in p): out.append("a lowercase letter")
    if not any(c.isdigit() for c in p): out.append("a number")
    if not any(not c.isalnum() for c in p): out.append("a special symbol")
    return out


def pw_error(problems):
    return "⚠️ Password needs " + ", ".join(problems) + "."


# ============================================================
# BRUTE-FORCE LOCKOUT  (stored in the DB, so refreshing the page does NOT reset it)
# ============================================================
def lock_until(key):
    with db() as c:
        r = c.execute("SELECT locked_until FROM lockouts WHERE key=?", (key,)).fetchone()
    return r[0] if r else 0


def register_failure(key):
    """Count a failed attempt. Returns (attempts_so_far, locked_now)."""
    now = time.time()
    with db() as c:
        r = c.execute("SELECT attempts, last_try FROM lockouts WHERE key=?", (key,)).fetchone()
        attempts = (r[0] if r and now - r[1] < 600 else 0) + 1     # stale attempts (>10 min) are forgotten
        if attempts >= MAX_ATTEMPTS:
            c.execute("INSERT OR REPLACE INTO lockouts VALUES (?,?,?,?)", (key, 0, now + LOCK_SECONDS, now))
            return MAX_ATTEMPTS, True
        c.execute("INSERT OR REPLACE INTO lockouts VALUES (?,?,?,?)", (key, attempts, 0, now))
        return attempts, False


def clear_failures(key):
    with db() as c:
        c.execute("DELETE FROM lockouts WHERE key=?", (key,))


def lock_countdown(key, title="Too many failed attempts"):
    """Show a live 30-second countdown, then rerun so the form is usable again."""
    box = st.empty()
    until = lock_until(key)
    while time.time() < until:
        left = max(1, int(until - time.time() + 0.99))
        box.markdown(H(f"""<div class="lock"><div class="t">{left}s</div>
            <div><b>🔒 {esc(title)}</b><span>Account locked. You can try again when the timer reaches zero.</span></div></div>"""),
                     unsafe_allow_html=True)
        time.sleep(1)
    box.empty()
    st.rerun()


def guard(key, title="Too many failed attempts"):
    if lock_until(key) > time.time():
        lock_countdown(key, title)


def on_fail(key, msg, title="Too many failed attempts"):
    n, locked = register_failure(key)
    if locked:
        lock_countdown(key, title)
    st.error(f"{msg} {MAX_ATTEMPTS - n} attempt(s) left before a {LOCK_SECONDS}-second lock.")


# ============================================================
# JWT SESSIONS
# ============================================================
def make_jwt(sub, role):
    now = datetime.datetime.now(datetime.timezone.utc)
    return jwt.encode({"sub": sub, "role": role, "iat": now, "exp": now + datetime.timedelta(hours=2)},
                      JWT_SECRET, algorithm="HS256")


def verify_jwt(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None


# ============================================================
# STATE / NAVIGATION
# ============================================================
DEFAULTS = {"token": None, "page": "Login", "flash": None, "reset_email": None, "reset_mode": None,
            "sq_text": None, "otp_code": None, "otp_expiry": None, "otp_verified": False}
for k, v in DEFAULTS.items():
    st.session_state.setdefault(k, v)


def navigate(p):
    st.session_state.page = p
    st.rerun()


def reset_flow():
    for k in ("reset_email", "reset_mode", "sq_text", "otp_code", "otp_expiry"):
        st.session_state[k] = None
    st.session_state.otp_verified = False


def logout():
    st.session_state.token = None
    st.session_state.page = "Login"
    st.rerun()


def mask_email(e):
    u, d = e.split("@", 1)
    return u[:2] + "•" * max(2, len(u) - 2) + "@" + d


# ============================================================
# OTP EMAIL
# ============================================================
def send_otp(email):
    sender, app_pw = os.environ.get("EMAIL_ADDRESS"), os.environ.get("EMAIL_PASSWORD")
    if not sender or not app_pw:
        return False, "Email service is not configured (EMAIL_ADDRESS / EMAIL_PASSWORD)."
    otp = f"{secrets.randbelow(10 ** 6):06d}"
    msg = EmailMessage()
    msg["Subject"] = f"{APP_NAME} - Password Reset OTP"
    msg["From"], msg["To"] = sender, email
    msg.set_content(f"Your password reset OTP is: {otp}\nIt expires in {OTP_TTL // 60} minutes. Do not share this code.")
    msg.add_alternative(H(f"""<div style="font-family:Arial,sans-serif;max-width:420px;margin:auto;border:1px solid #e2e8f0;border-radius:16px;padding:28px;text-align:center;">
        <div style="font-size:34px;">⚓</div><h2 style="margin:6px 0;color:#0f172a;">{APP_NAME}</h2>
        <p style="color:#64748b;">Use this code to reset your password</p>
        <div style="font-size:34px;font-weight:800;letter-spacing:8px;color:#0369a1;background:#e0f2fe;border-radius:12px;padding:14px;">{otp}</div>
        <p style="color:#64748b;font-size:13px;">Valid for {OTP_TTL // 60} minutes. If you didn't request this, ignore this email.</p></div>"""),
                          subtype="html")
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=20) as s:
            s.login(sender, app_pw.replace(" ", ""))
            s.send_message(msg)
    except Exception:
        return False, "Could not send the OTP email. Check the Gmail App Password and try again."
    st.session_state.update(reset_email=email, reset_mode="otp", otp_code=otp,
                            otp_expiry=time.time() + OTP_TTL, otp_verified=False)
    return True, ""


# ============================================================
# UI PIECES
# ============================================================
FEATURES = [("💲", "Agentic freight pricing", "AI agents forecast rates and quote shipments."),
            ("🧭", "Route optimization", "Fuel- and time-efficient sea routes between ports."),
            ("🔐", "Secure access", "JWT sessions, OTP recovery and 30-second brute-force lock.")]

_SCENE = ("<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 520 190'>"
          "<path d='M14 52 C120 -4 230 100 340 48 S470 14 506 58' fill='none' stroke='#67e8f9' stroke-width='2' stroke-dasharray='2 8' stroke-linecap='round'/>"
          "<circle cx='14' cy='52' r='6' fill='#fcd34d'/><circle cx='506' cy='58' r='6' fill='#34d399'/>"
          "<g transform='translate(150 70)'>"
          "<rect x='20' y='22' width='26' height='16' fill='#f97316'/><rect x='48' y='22' width='26' height='16' fill='#38bdf8'/>"
          "<rect x='76' y='22' width='26' height='16' fill='#facc15'/><rect x='104' y='22' width='26' height='16' fill='#f43f5e'/>"
          "<rect x='34' y='8' width='26' height='14' fill='#34d399'/><rect x='62' y='8' width='26' height='14' fill='#f97316'/>"
          "<rect x='90' y='8' width='26' height='14' fill='#38bdf8'/>"
          "<rect x='136' y='4' width='22' height='34' fill='#e2e8f0'/><rect x='140' y='11' width='14' height='6' fill='#0ea5e9'/>"
          "<rect x='145' y='-4' width='4' height='8' fill='#e2e8f0'/>"
          "<path d='M6 38 H172 L156 60 H24Z' fill='#0f172a'/><path d='M6 38 H172 L169 43 H9Z' fill='#ef4444'/></g>"
          "<path d='M0 132 C40 116 80 116 120 132 S200 148 240 132 S320 116 360 132 S440 148 480 132 S510 120 520 128 V190 H0Z' fill='#06b6d4' fill-opacity='.38'/>"
          "<path d='M0 152 C40 138 80 138 120 152 S200 166 240 152 S320 138 360 152 S440 166 480 152 S510 142 520 148 V190 H0Z' fill='#0e7490' fill-opacity='.65'/>"
          "<path d='M0 172 C60 162 120 162 180 172 S300 182 360 172 S470 162 520 172 V190 H0Z' fill='#082f49'/></svg>")
_SCENE_URI = "data:image/svg+xml;utf8," + quote(_SCENE, safe="/:=;,'")


def hero(lead):
    feats = "".join(f'<div class="feat"><div class="ic">{i}</div><div><b>{t}</b><span>{d}</span></div></div>'
                    for i, t, d in FEATURES)
    st.markdown(H(f"""<div class="hero"><div class="brand"><div class="logo">⚓</div>{APP_NAME}</div>
        <div class="chip">Infosys Springboard 7.0 · Milestone 1</div>
        <h1>Agentic AI for <em>Maritime Freight Pricing</em> and Route Optimization</h1>
        <p class="lead">{lead}</p>{feats}
        <img class="scene" src="{_SCENE_URI}" alt=""></div>"""), unsafe_allow_html=True)


def card_title(title, sub):
    st.markdown(H(f'<div class="card-title">{esc(title)}</div><div class="card-sub">{esc(sub)}</div>'),
                unsafe_allow_html=True)


def show_flash():
    if st.session_state.flash:
        st.success(st.session_state.flash)
        st.session_state.flash = None


def finish_reset(email, npw, cpw):
    """Shared last step of both recovery routes."""
    problems = password_problems(npw)
    if not npw or not cpw:
        st.error("⚠️ Please fill in both password fields.")
    elif problems:
        st.error(pw_error(problems))
    elif npw != cpw:
        st.error("❌ Passwords do not match.")
    elif password_taken(npw, exclude_email=email):
        st.error("❌ This password is not available. Please choose a different one.")
    else:
        try:
            with db() as c:
                c.execute("UPDATE users SET password_hash=?, pwd_fp=? WHERE email=?", (hash_txt(npw), pwd_fp(npw), email))
        except sqlite3.IntegrityError:
            st.error("❌ This password is not available. Please choose a different one.")
            return
        clear_failures("login:" + email)
        reset_flow()
        st.session_state.flash = "✅ Password updated. Please sign in with your new password."
        navigate("Login")


# ============================================================
# PAGES — NOT LOGGED IN
# ============================================================
if not st.session_state.token:
    # no sidebar on the auth pages
    st.markdown("<style>section[data-testid='stSidebar'],[data-testid='stSidebarCollapsedControl'],[data-testid='stExpandSidebarButton']{display:none !important;}</style>",
                unsafe_allow_html=True)
    st.markdown(f"<style>{AUTH_BG_CSS}</style>", unsafe_allow_html=True)
    if st.session_state.page not in ("Login", "Signup", "Forgot", "AdminLogin"):
        st.session_state.page = "Login"
    page = st.session_state.page

    left, right = st.columns([1.1, 1], gap="large", vertical_alignment="center")
    with left:
        if page == "Signup":
            hero("Pick a username, a strong unique password and a security question for account recovery.")
        elif page == "Forgot":
            hero("Recover access with your security question or a one-time code sent to your registered email.")
        elif page == "AdminLogin":
            hero("Restricted area. Sign in with the administrator credentials to view registered users.")
        else:
            hero("Sign in to quote freight, compare sea routes and track optimization results — protected by modern authentication.")

    with right:
        show_flash()

        # ---------------- LOGIN ----------------
        if page == "Login":
            with st.form("login_form"):
                card_title("Welcome back 👋", "Sign in to your account to continue")
                ident = st.text_input("Username or email", placeholder="username or you@example.com")
                pwd = st.text_input("Password", type="password", placeholder="Enter your password")
                submitted = st.form_submit_button("Sign in  →", type="primary", width="stretch")
            if submitted:
                ident = ident.strip()
                if not ident or not pwd:
                    st.error("⚠️ Username/email and password are required.")
                else:
                    user = find_user(ident)
                    key = "login:" + (user["email"] if user else ident.lower())
                    guard(key)
                    ok = check_txt(pwd, user["password_hash"] if user else dummy_hash()) and user is not None
                    if ok:
                        clear_failures(key)
                        if not user.get("pwd_fp"):          # backfill fingerprint for older accounts
                            try:
                                with db() as c:
                                    c.execute("UPDATE users SET pwd_fp=? WHERE id=?", (pwd_fp(pwd), user["id"]))
                            except sqlite3.IntegrityError:
                                pass
                        st.session_state.token = make_jwt(user["email"], "user")
                        navigate("Dashboard")
                    else:
                        on_fail(key, "❌ Invalid credentials.", "Too many failed sign-in attempts")
            c1, c2, c3 = st.columns(3)
            if c1.button("Create account", width="stretch"): navigate("Signup")
            if c2.button("Forgot password?", width="stretch"): navigate("Forgot")
            if c3.button("🛡️ Admin login", width="stretch"): navigate("AdminLogin")

        # ---------------- SIGNUP ----------------
        elif page == "Signup":
            with st.form("signup_form"):
                card_title("Create account", f"Join {APP_NAME} — all fields are required")
                uname = st.text_input("Username", placeholder="e.g. jane.doe")
                email = st.text_input("Email address", placeholder="you@example.com")
                p1, p2 = st.columns(2)
                pwd = p1.text_input("Password", type="password", placeholder="Create a password")
                cpw = p2.text_input("Confirm password", type="password", placeholder="Re-enter password")
                st.caption("Password: 8+ characters with an uppercase letter, lowercase letter, number and special symbol. "
                           "It must be different from every other user's password.")
                sq = st.selectbox("Security question", SECURITY_QUESTIONS)
                sa = st.text_input("Security answer", placeholder="Your answer")
                submitted = st.form_submit_button("Create account  →", type="primary", width="stretch")
            if submitted:
                uname, email, sa = uname.strip(), email.strip().lower(), sa.strip()
                problems = password_problems(pwd)
                if not all([uname, email, pwd, cpw, sa]):
                    st.error("⚠️ Please fill in all required fields.")
                elif not valid_username(uname):
                    st.error("❌ Username must be 3–30 characters (letters, numbers, space, . _ -).")
                elif not valid_email(email):
                    st.error("❌ Enter a valid email address (e.g. ab@cd.ef).")
                elif problems:
                    st.error(pw_error(problems))
                elif pwd != cpw:
                    st.error("❌ Passwords do not match.")
                else:
                    with db() as c:
                        name_taken = c.execute("SELECT 1 FROM users WHERE lower(username)=lower(?)", (uname,)).fetchone()
                        mail_taken = c.execute("SELECT 1 FROM users WHERE email=?", (email,)).fetchone()
                    if name_taken:
                        st.error("❌ That username is already taken. Please choose another.")
                    elif mail_taken:
                        st.error("❌ That email is already registered.")
                    elif password_taken(pwd):
                        st.error("❌ This password is not available. Please choose a different one.")
                    else:
                        try:
                            with db() as c:
                                c.execute("INSERT INTO users (username,email,password_hash,security_question,security_answer_hash,pwd_fp) VALUES (?,?,?,?,?,?)",
                                          (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower()), pwd_fp(pwd)))
                        except sqlite3.IntegrityError:
                            st.error("❌ Username, email or password is not available. Please change it and try again.")
                        else:
                            st.session_state.flash = "✅ Account created! Please sign in."
                            navigate("Login")
            if st.button("← Back to sign in", width="stretch"): navigate("Login")

        # ---------------- FORGOT PASSWORD ----------------
        elif page == "Forgot":
            mode, remail = st.session_state.reset_mode, st.session_state.reset_email

            if not mode:   # step 1: pick the route
                with st.form("forgot_start"):
                    card_title("Reset password", "Verify it's you with a security question or an email OTP")
                    ident = st.text_input("Username or registered email", placeholder="username or you@example.com")
                    b1, b2 = st.columns(2)
                    sq_btn = b1.form_submit_button("Security question", type="secondary", width="stretch")
                    otp_btn = b2.form_submit_button("Email OTP", type="primary", width="stretch")
                if sq_btn or otp_btn:
                    user = find_user(ident.strip()) if ident.strip() else None
                    if not ident.strip():
                        st.error("⚠️ Enter your username or registered email.")
                    elif not user:
                        st.error("❌ No account found with that username or email.")
                    elif sq_btn:
                        st.session_state.update(reset_email=user["email"], reset_mode="sq", sq_text=user["security_question"])
                        st.rerun()
                    else:
                        guard("otp:" + user["email"], "Too many wrong OTP attempts")
                        ok, err = send_otp(user["email"])
                        if ok: st.rerun()
                        else: st.error("❌ " + err)

            elif mode == "sq":   # route A: security question
                key = "sq:" + remail
                with st.form("forgot_sq"):
                    card_title("Security question", "Answer correctly, then choose a new password")
                    st.info(f"❓ {st.session_state.sq_text}")
                    ans = st.text_input("Your answer")
                    n1, n2 = st.columns(2)
                    npw = n1.text_input("New password", type="password")
                    cpw = n2.text_input("Confirm new password", type="password")
                    go_btn = st.form_submit_button("Reset password  →", type="primary", width="stretch")
                if go_btn:
                    if not ans.strip() or not npw or not cpw:
                        st.error("⚠️ All fields are required.")
                    else:
                        guard(key, "Too many wrong answers")
                        with db() as c:
                            r = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (remail,)).fetchone()
                        if r and check_txt(ans.strip().lower(), r[0]):
                            clear_failures(key)
                            finish_reset(remail, npw, cpw)
                        else:
                            on_fail(key, "❌ Incorrect security answer.", "Too many wrong answers")

            elif mode == "otp" and not st.session_state.otp_verified:   # route B, step 1: verify OTP
                key = "otp:" + remail
                with st.form("forgot_otp"):
                    card_title("Enter OTP", f"We emailed a 6-digit code to {mask_email(remail)}")
                    entered = st.text_input("6-digit OTP", max_chars=6, placeholder="••••••")
                    verify = st.form_submit_button("Verify OTP  →", type="primary", width="stretch")
                if verify:
                    guard(key, "Too many wrong OTP attempts")
                    if not st.session_state.otp_code:
                        st.error("⚠️ No active OTP. Click “Resend OTP” to get a new one.")
                    elif time.time() > st.session_state.otp_expiry:
                        st.session_state.otp_code = None
                        st.error("⌛ OTP expired. Click “Resend OTP” to get a new one.")
                    elif not entered.strip():
                        st.error("⚠️ Enter the OTP.")
                    elif secrets.compare_digest(entered.strip().encode(), st.session_state.otp_code.encode()):
                        clear_failures(key)
                        st.session_state.otp_verified = True
                        st.rerun()
                    else:
                        n, locked = register_failure(key)
                        if locked:
                            st.session_state.otp_code = None     # the old code is dead; a new one is needed
                            lock_countdown(key, "Too many wrong OTP attempts")
                        st.error(f"❌ Incorrect OTP. {MAX_ATTEMPTS - n} attempt(s) left before a {LOCK_SECONDS}-second lock.")
                if st.button("↻ Resend OTP", width="stretch"):
                    guard(key, "Too many wrong OTP attempts")
                    ok, err = send_otp(remail)
                    if ok:
                        st.success("📧 A new OTP has been sent.")
                    else:
                        st.error("❌ " + err)
                if not st.session_state.otp_code:
                    st.caption("The previous OTP is no longer valid — request a new one.")

            elif mode == "otp":   # route B, step 2: new password
                with st.form("forgot_newpw"):
                    card_title("Set a new password", "OTP verified ✅ — choose a strong password")
                    n1, n2 = st.columns(2)
                    npw = n1.text_input("New password", type="password")
                    cpw = n2.text_input("Confirm new password", type="password")
                    go_btn = st.form_submit_button("Reset password  →", type="primary", width="stretch")
                if go_btn:
                    if time.time() > (st.session_state.otp_expiry or 0):
                        st.session_state.otp_verified = False
                        st.session_state.otp_code = None
                        st.error("⌛ OTP expired. Please start again.")
                    else:
                        finish_reset(remail, npw, cpw)

            if st.button("← Cancel and go back", width="stretch"):
                reset_flow()
                navigate("Login")

        # ---------------- ADMIN LOGIN ----------------
        elif page == "AdminLogin":
            with st.form("admin_form"):
                card_title("Admin sign in 🛡️", "Restricted to administrators")
                a_user = st.text_input("Admin username")
                a_pass = st.text_input("Admin password", type="password")
                submitted = st.form_submit_button("Sign in as admin  →", type="primary", width="stretch")
            if submitted:
                if not a_user.strip() or not a_pass:
                    st.error("⚠️ Admin username and password are required.")
                elif not ADMIN_USERNAME or not ADMIN_PASSWORD:
                    st.error("Admin credentials are not configured (ADMIN_USERNAME / ADMIN_PASSWORD).")
                else:
                    key = "admin:" + a_user.strip().lower()
                    guard(key, "Too many failed admin attempts")
                    ok_u = hmac.compare_digest(a_user.strip().encode(), ADMIN_USERNAME.encode())
                    ok_p = hmac.compare_digest(a_pass.encode(), ADMIN_PASSWORD.encode())
                    if ok_u and ok_p:
                        clear_failures(key)
                        st.session_state.token = make_jwt(ADMIN_USERNAME, "admin")
                        navigate("Dashboard")
                    else:
                        on_fail(key, "❌ Invalid admin credentials.", "Too many failed admin attempts")
            if st.button("← Back to user sign in", width="stretch"): navigate("Login")

# ============================================================
# PAGES — LOGGED IN
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        logout()
    role = payload.get("role")

    # ---------------- ADMIN DASHBOARD ----------------
    if role == "admin":
        with st.sidebar:
            st.markdown(H(f"""<div style="text-align:center;padding:14px 4px;"><div style="font-size:30px;">🛡️</div>
                <div style="font-weight:800;font-size:16px;">Admin Console</div>
                <div style="font-size:12px;color:#64748b;">{APP_NAME}</div></div>"""), unsafe_allow_html=True)
            menu = option_menu(None, ["Users", "Logout"], icons=["people", "box-arrow-right"], key="menu_admin",
                               styles={"container": {"background-color": "#ffffff"},
                                       "nav-link-selected": {"background-color": "#0e7490", "color": "#fff"}})
            if menu == "Logout":
                logout()

        st.markdown(H(f"""<div class="banner"><div><h1>🛡️ Admin Dashboard</h1>
            <div class="sub">{esc(PROJECT_TITLE)} · registered users (password data is never shown)</div></div>
            <div class="pill">🛡️ {esc(payload.get('sub'))}</div></div>"""), unsafe_allow_html=True)

        admin_msg = st.session_state.pop("admin_flash", None)
        if admin_msg:
            st.success(admin_msg)

        with db() as c:
            rows = c.execute("SELECT id, username, email, security_question FROM users ORDER BY id").fetchall()
            locked = {k for (k,) in c.execute("SELECT key FROM lockouts WHERE key LIKE 'login:%' AND locked_until > ?", (time.time(),))}
        df = pd.DataFrame(rows, columns=["ID", "Username", "Email", "Security question"])
        df["Status"] = df["Email"].map(lambda e: "🔒 Locked" if "login:" + e in locked else "✅ Active")

        k1, k2, k3 = st.columns(3)
        for col, ic, v, l in [(k1, "👥", len(df), "Registered users"), (k2, "🔒", int((df["Status"] != "✅ Active").sum()), "Locked right now"),
                              (k3, "🔑", "bcrypt", "Password storage")]:
            col.markdown(H(f'<div class="kpi"><div class="ic">{ic}</div><div class="v">{v}</div><div class="l">{l}</div></div>'),
                         unsafe_allow_html=True)
        st.markdown("<br>", unsafe_allow_html=True)
        q = st.text_input("Search users", placeholder="Filter by username or email")
        if q.strip():
            m = df["Username"].str.contains(q.strip(), case=False, regex=False) | df["Email"].str.contains(q.strip(), case=False, regex=False)
            df = df[m]
        st.dataframe(df, width="stretch", hide_index=True)

        # ---------- remove registered users ----------
        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown("#### 🗑️ Remove registered users")
        labels = {f"{r[1]}  ·  {r[2]}": r[0] for r in rows}        # always lists ALL users (ignores the search box)
        if not labels:
            st.info("No registered users to remove.")
        else:
            with st.form("admin_delete_form"):
                picked = st.multiselect("Select the user(s) to remove", list(labels), placeholder="Choose one or more users")
                sure = st.checkbox("I understand this permanently deletes the selected account(s) and cannot be undone.")
                del_btn = st.form_submit_button("Delete selected users", type="primary", width="stretch")
            if del_btn:
                if not picked:
                    st.error("⚠️ Select at least one user.")
                elif not sure:
                    st.error("⚠️ Tick the confirmation box before deleting.")
                else:
                    ids = [labels[p] for p in picked]
                    marks = ",".join("?" * len(ids))
                    with db() as c:
                        emails = [r[0] for r in c.execute(f"SELECT email FROM users WHERE id IN ({marks})", ids)]
                        c.execute(f"DELETE FROM users WHERE id IN ({marks})", ids)
                        for e in emails:                           # also forget their lock-out counters
                            c.execute("DELETE FROM lockouts WHERE key IN (?,?,?)", ("login:" + e, "otp:" + e, "sq:" + e))
                    st.session_state.admin_flash = f"✅ Removed {len(ids)} user(s): " + ", ".join(p.split("  ·  ")[0] for p in picked)
                    st.rerun()

    # ---------------- USER DASHBOARD ----------------
    else:
        user = find_user(payload.get("sub", ""))
        if not user:
            logout()
        uname, email = user["username"], user["email"]

        with st.sidebar:
            st.markdown(H(f"""<div style="text-align:center;padding:14px 4px;"><div style="font-size:30px;">⚓</div>
                <div style="font-weight:800;font-size:16px;">{APP_NAME}</div>
                <div style="font-size:12px;color:#64748b;">Freight pricing &amp; routing</div></div>"""), unsafe_allow_html=True)
            menu = option_menu(None, ["Dashboard", "Analytics", "Reports", "Logout"],
                               icons=["house", "graph-up", "file-text", "box-arrow-right"], key="menu_user",
                               styles={"container": {"background-color": "#ffffff"},
                                       "nav-link-selected": {"background-color": "#0e7490", "color": "#fff"}})
            if menu == "Logout":
                logout()

        st.markdown(H(f"""<div class="banner"><div><h1>Welcome back, {esc(uname)} 👋</h1>
            <div class="sub">{esc(PROJECT_TITLE)} · signed in as {esc(email)}</div></div><div class="pill">👤 {esc(uname)}</div></div>"""),
                    unsafe_allow_html=True)

        cols = st.columns(4)
        for col, ic, lbl, val in [(cols[0], "🚢", "Active Routes", "128"), (cols[1], "💲", "Quotes Today", "47"),
                                  (cols[2], "⛽", "Avg. Fuel Saving", "12.4%"), (cols[3], "🛡️", "Security Status", "Secured")]:
            col.markdown(H(f'<div class="kpi"><div class="ic">{ic}</div><div class="v">{val}</div><div class="l">{lbl}</div></div>'),
                         unsafe_allow_html=True)
        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92,
                                     title={"text": "Route Optimization Index", "font": {"color": "#0f172a", "size": 14}},
                                     gauge={"axis": {"range": [0, 100]}, "bar": {"color": "#0891b2"}, "bgcolor": "#e0f2fe",
                                            "borderwidth": 0}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": "#334155", "family": "Inter"}, height=270,
                          margin=dict(l=10, r=10, t=50, b=10))
        st.plotly_chart(fig, width="stretch")


Writing app.py


### 4️⃣ Launch the app (ngrok public URL)

In [ ]:
import os, time, subprocess
from pyngrok import ngrok, conf

if "DB_PATH" not in os.environ or "JWT_SECRET" not in os.environ:
    raise RuntimeError("Run the setup cell first (it loads your Colab Secrets and the database path).")

# Pass the token for this session only. (ngrok.set_auth_token() fails with
# "Authtoken already set" when a previous run already saved one.)
conf.get_default().auth_token = os.environ["NGROK_AUTHTOKEN"]

# Stop anything left over from a previous run (a leftover tunnel keeps your free ngrok URL
# 'busy' and causes ERR_NGROK_334 "endpoint is already online")
for t in ngrok.get_tunnels():
    try:
        ngrok.disconnect(t.public_url)
    except Exception:
        pass
ngrok.kill()
!pkill -x ngrok
!pkill -f streamlit
time.sleep(3)   # give ngrok a moment to release the URL

# Start Streamlit in the background (it inherits the secrets from os.environ)
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

public_url = ngrok.connect(8501, "http").public_url
print("=" * 60)
print(f"🚀 Infosys Portal Live URL: {public_url}")
print("=" * 60)
print("⏳ App is running! Press the Colab Stop button to shut down.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ Ngrok tunnel closed and Streamlit server stopped.")


🚀 Infosys Portal Live URL: https://figurine-stillness-hertz.ngrok-free.dev
⏳ App is running! Press the Colab Stop button to shut down.


### Optional — check active tunnels

In [ ]:
from pyngrok import ngrok

# Optional: list the active tunnels in this Colab session
print(ngrok.get_tunnels())
